In [ ]:
# 1) Extract all numbers from text
import re
text = "SKU-12 qty=5 price=399.50"
re.findall(r"\d+")

- re.findall(pattern, text)
- scans the entire text, finds all non-overlapping matches of the regex pattern, returns a list of matches (usually a atring)

- r"\d+(?:\.\d+)?"
- r means raw string, so Python does not treat backlashes as escapes
- \d means a digit (0-9)
- (+) means one or more

So, \d+ matches: 12, 5, 399, 0 But not an empty string

- Part B: (?:....)(non-capturing group)
- (?:\.\d+) groups things together without capturing them separately/ to apply to whole decimal part, not just one character
- (?:...) = group for logic, but don't return it separately

- \.\d+
- \. means dot (dot. in regex means any character)
- so we escape it as \. to mean actual decimal point
- \d+ again means one or more digits after the dot

- so \.\d+ matches
- 0.5 / 0.0 / 0.123

- (?:\.\d+)?
- ? means preceding thing occurs 0 or 1 time
- Here the preceding thing is entire decimal-part group.
- So decimal part is optional:
- 399 okay
- 399.5 ok

1) Extract all numbers from text

In [1]:
import re
text = "SKU-12 qty=5 price=399.50"
num = re.findall(r"\d+(?:\.\d+)?", text)
print(num)

['12', '5', '399.50']


2) Validate Indian PIN code (6 digits, not starting with 0)

In [6]:
pins = ["560001", "012345", "56001", "5600011", "ABCDE1"]
pattern = r"^[1-9]\d{5}$"
for p in pins:
    print(p, bool(re.fullmatch(pattern,p)))


560001 True
012345 False
56001 False
5600011 False
ABCDE1 False


- ^ ensure matching starts from the beginning
- [1-9] first digit
- \d{5} - next 5 digits
- \d - any digit 0-9
- {5} exactly 5 times
- $ ensures matching end exactly here

3) Extract email IDs from a message

In [7]:
import re
text = "Mail: paritosh@gmail.com, support@company.in, wrong@mail, a@b.c"
emails = re.findall(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9._]+\.[A-Za-z]{2,}", text)
print(emails)

['paritosh@gmail.com', 'support@company.in']


[A-Za-z0-9._%+-]+   @   [A-Za-z0-9._]+   \.   [A-Za-z]{2,}


   (username)      @       (domain)     dot     (extension)

- {2,} means at least 2 letters

4) Parse key=value pairs from a log line into a dict

In [8]:
import re
line = "ts=2026-05-07 level=ERROR user=AB12 msg=timeout retry=3"
pairs = re.findall(r"(\w+)=([^\s]+)", line)
data = dict(pairs)
print(data)


{'ts': '2026-05-07', 'level': 'ERROR', 'user': 'AB12', 'msg': 'timeout', 'retry': '3'}


- re.findall(pattern, line) scans the full string and returns all matches of the pattern.
- Because your pattern has two capturing groups ( ... ), findall() returns a list of tuples like:

In [12]:
import re
line = "ts=2026-05-07 level=ERROR user=AB12 msg=timeout retry=3"
print(re.findall(r"\w+=\S+", line))

['ts=2026-05-07', 'level=ERROR', 'user=AB12', 'msg=timeout', 'retry=3']


- w/o the () we get the whole chunk, but key, values are not separated

5) Parse a structured log with separators (timestamp | level | user= | msg=)

In [15]:
import re
line = "2026-05-07 10:30:12 | ERROR | user=AB12 | msg=timeout while calling API"
pat = (
    r"^(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s*\|\s*"
    r"(?P<level>\w+)\s*\|\s*user=(?P<user>\w+)\s*\|\s*msg=(?P<msg>.+)$"
)

m = re.search(pat, line)
print(m.groupdict())
# {'ts': '2026-05-07 10:30:12', 'level': 'ERROR', 'user': 'AB12', 'msg': 'timeout while calling API'}

{'ts': '2026-05-07 10:30:12', 'level': 'ERROR', 'user': 'AB12', 'msg': 'timeout while calling API'}


- ^ means matching must start at the beginning of the line/ so random text before timestamp won't be allowed
- (?P<ts>....) - capture whatever matches inside and name it ts
- Inside it \d{4}- 4 digits (year) - 2026
- \d{2} - months and date
- space
- time
- So ts becomes 2026-05-07 10:30:12

- \s* - 0 or more spaces/tabs
- \| - literal pipe |
- \s* again optional spaces
- so it matches any of these safely
- " | "
- "|"
- " | "

 - (?P<level>\w+)
  - \w+ → one or more “word chars” (letters/digits/underscore)
    -  this captures thing like ERROR,INFO

- \s*\|\s* one or more spaces with pipe
- (?P<level>\w+)
- \w+ → one or more “word chars” (letters/digits/underscore)
- This captures things like: ERROR, INFO, WARN
- So level becomes ERROR

- user=(?P<user>\w+) — user field

6) Replace multiple spaces/tabs/newlines with a single space (text cleaning)

In [16]:
text = "Hello   \t world\nThis   is   Python"
import re

clean = re.sub(r"\s+", " ", text).strip()
print(clean)  # "Hello world This is Python"

Hello world This is Python


7) Split text by multiple delimiters (comma, semicolon, pipe)

In [17]:
text = "A,B;C|D  ,  E"
import re

parts = [x.strip() for x in re.split(r"[,\;|]+", text) if x.strip()]
print(parts)  # ['A', 'B', 'C', 'D', 'E']

['A', 'B', 'C', 'D', 'E']


8) Extract words starting with a capital letter (simple entity extraction)

In [18]:
text = "Paritosh works at UltraTech Cement in Bangalore."
import re

caps = re.findall(r"\b[A-Z][a-z]+\b", text)
print(caps)  # ['Paritosh', 'UltraTech', 'Cement', 'Bangalore']

['Paritosh', 'Cement', 'Bangalore']


9) Normalize dates: accept YYYY-MM-DD or DD-MM-YYYY → output YYYY-MM-DD

In [19]:
dates = ["2025-11-01", "29-05-2025", "07-01-2026"]
import re

def normalize_date(s: str) -> str:
    # YYYY-MM-DD
    m1 = re.fullmatch(r"(\d{4})-(\d{2})-(\d{2})", s)
    if m1:
        y, m, d = m1.groups()
        return f"{y}-{m}-{d}"

    # DD-MM-YYYY
    m2 = re.fullmatch(r"(\d{2})-(\d{2})-(\d{4})", s)
    if m2:
        d, m, y = m2.groups()
        return f"{y}-{m}-{d}"

    raise ValueError(f"Unsupported date format: {s}")

for d in dates:
    print(d, "->", normalize_date(d))

2025-11-01 -> 2025-11-01
29-05-2025 -> 2025-05-29
07-01-2026 -> 2026-01-07


10) Extract “Status” mapping from numeric codes (parsing + business logic)

In [20]:
statuses = [109, 110, 111, 999]
def map_status(code: int) -> str:
    return {
        109: "Converted",
        110: "Marked as not converted",
        111: "Open"
    }.get(code, "Unknown")

for s in statuses:
    print(s, "->", map_status(s))

109 -> Converted
110 -> Marked as not converted
111 -> Open
999 -> Unknown
